In [ ]:
%load_ext autoreload
%autoreload 2

### Run MHCPrime and log odds on datasets

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "CLSBN26":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

In [ ]:
# imports
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.log_odds import *
from src.plotting import *

# misc
import warnings
warnings.filterwarnings("ignore")
device="cuda"

In [ ]:
# change gpu
GPU_ID = 0
# GPU_ID = 1
# GPU_ID = 2
# GPU_ID = 3

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")

Using GPU: NVIDIA H100 80GB HBM3 (ID: 0)


### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
from Data.mhc_pseudo_seqs import mhc_seq_dict
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### Datasets

In [ ]:
# ms (already run in other notebooks)
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data.csv", index_col=0) # get for log odds
# ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

# yd test set
holec_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "Holec_et_al_2025"
holec_test_set  = pd.read_csv(f"{holec_datasets_dir}/holec_test_set.csv", index_col=0)

# episcan train and test sets
episcan_train_test_dir = DATA_DIR / "Training_Datasets" / "Processed" / "EpiScan"
episcan_r9_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv", index_col=0)
episcan_rt_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv", index_col=0)
sd3_formatted=pd.read_csv(f"{episcan_train_test_dir}/sd3_formatted.csv", index_col=0)

# ba test sets
ba_full_cls_test_set=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_cls_test_set.csv", index_col=0)
ba_full_con_test_set=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_con_test_set.csv", index_col=0)
nmp_ba_data_hla=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/nmp_ba_data_hla.csv", index_col=0)

# escape seq test sets
escape_seq_datasets_dir = DATA_DIR / "Training_Datasets/Processed/ESCAPE_seq"
escape_seq_full_cls_test_set=pd.read_csv(f"{escape_seq_datasets_dir}/escape_seq_full_cls_test_set.csv", index_col=0)
escape_seq_full_con_test_set=pd.read_csv(f"{escape_seq_datasets_dir}/escape_seq_full_con_test_set.csv", index_col=0)

In [ ]:
%%time
# process all
ms_train_data=full_process_data(ms_train_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_train_data=create_seq_allele_column(ms_train_data)

holec_test_set=full_process_data(holec_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
holec_test_set=create_seq_allele_column(holec_test_set)

episcan_r9_test_set=full_process_data(episcan_r9_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
episcan_r9_test_set=create_seq_allele_column(episcan_r9_test_set)

episcan_rt_test_set=full_process_data(episcan_rt_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
episcan_rt_test_set=create_seq_allele_column(episcan_rt_test_set)

sd3_formatted=full_process_data(sd3_formatted, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
sd3_formatted=create_seq_allele_column(sd3_formatted)

ba_full_cls_test_set=full_process_data(ba_full_cls_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ba_full_cls_test_set=create_seq_allele_column(ba_full_cls_test_set)

ba_full_con_test_set=full_process_data(ba_full_con_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ba_full_con_test_set=create_seq_allele_column(ba_full_con_test_set)

nmp_ba_data_hla=full_process_data(nmp_ba_data_hla, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
nmp_ba_data_hla=create_seq_allele_column(nmp_ba_data_hla)

escape_seq_full_cls_test_set=full_process_data(escape_seq_full_cls_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
escape_seq_full_cls_test_set=create_seq_allele_column(escape_seq_full_cls_test_set)

escape_seq_full_con_test_set=full_process_data(escape_seq_full_con_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
escape_seq_full_con_test_set=create_seq_allele_column(escape_seq_full_con_test_set)

CPU times: user 36 s, sys: 6.66 s, total: 42.7 s
Wall time: 42.6 s


In [ ]:
# human proteome
canonical_human_proteins=pd.read_csv(f"{DATA_DIR}/canonical_human_proteins.csv", index_col=0)

### Load MHCPrime

In [ ]:
# setup tokenizer
tokenizer = AminoAcidTokenizer()

In [ ]:
# set base model params
base_model_params={
        'vocab_size': tokenizer.vocab_size,
        'embed_dim': 480,
        'fc_hidden_dims': [240, 120], 
        'fc_dropout': 0.1,
        'n_heads': 16,
        'max_peptide_len': 34,
        'max_mhc_len': 34,
        "n_peptide_layers": 2,
        "n_mhc_layers": 1,
        'n_concat_layers': 3,
        'add_pos_enc': True,
        'use_segment_embedder': True,
        'dim_feedforward_multiplier': 1,
        'debugging': False,
        'fc_output_dim': 1,
        'pooling': 'cls', # cls, mean, min, max, attn, for sequence ffnn head
        "ffnn_use_layernorm": False, 
        'use_feature_table': True,
        "feature_table": processed_feature_table,
        "sequence_embedder_feature_names": feature_names,
        'sequence_embedder_feature_mode': "concat",  # replace or concat
        'sequence_embedder_tokenizer': tokenizer,
        "peptide_start_pos": 1, 
        "mhc_start_pos": 35, 
        "pos_enc_type": "sincos", 
        "seed": 42,

        # peptide conformer
        "add_peptide_conformer": True, 
        "peptide_conformer_num_layers": 2,
        "peptide_conformer_n_heads": 16,
        "peptide_conformer_conv_kernel": 5,
        "peptide_conformer_dropout": 0.0,

        # mhc conformer
        "add_mhc_conformer": False, 
        "mhc_conformer_num_layers": 2,
        "mhc_conformer_n_heads": 16,
        "mhc_conformer_conv_kernel": 5,
        "mhc_conformer_dropout": 0.0,

        # fusion conformer parameters
        "add_fusion_conformer": True, 
        "fusion_conformer_num_layers": 1,
        "fusion_conformer_n_heads": 16,
        "fusion_conformer_conv_kernel": 7,
        "fusion_conformer_dropout": 0.0,
        "fusion_conformer_mask_bos": True,
        "ff_pre_post_swiglu": False,

        # legacy init
        "legacy_init_compat": True,
}

In [ ]:
%%time
# load base model (optionally, can load model_params directly)
# run_name = "MHCPrime_Base"
# run_dir = MODEL_CHECKPOINTS_DIR / run_name
# model_dir = run_dir / "Checkpoints"

# mhcprime=init_model(base_model_params, print_check=False, print_params=True)
# mhcprime.load_state_dict(torch.load(f"{model_dir}/model_final.pt"))
# mhcprime.eval()
# print("Model loaded from checkpoint...")
# clear_all_gpu_memory()

base_model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Base" / "Checkpoints"
ckpt_path = base_model_dir / "model_final.pt"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

clear_all_gpu_memory()

mhcprime = init_model(
    base_model_params,
    print_check=False,
    print_params=True,
)

state_dict = torch.load(
    ckpt_path,
    map_location="cpu",
)

mhcprime.load_state_dict(state_dict, strict=True)
mhcprime.to(device)
mhcprime.eval()

print("Model loaded from checkpoint...")
print(f"Checkpoint: {ckpt_path}")
print(f"Model device: {next(mhcprime.parameters()).device}")

[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
All GPU memory has been cleared.
Current GPU memory allocated: 0.10 GB
Current GPU memory reserved: 0.12 GB
CPU times: user 908 ms, sys: 717 ms, total: 1.62 s
Wall time: 1.74 s


### Inference

In [ ]:
# run mhcprime
base_and_ft_models={
    "mhcprime": mhcprime,
}

In [ ]:
%%time
# holec_test_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(holec_test_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    holec_test_set[model_name]=output_scores.detach().cpu().tolist()

160000it [00:04, 34948.06it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (43.5 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 53/53 [00:10<00:00,  4.92it/s]


CPU times: user 2min 57s, sys: 4.65 s, total: 3min 2s
Wall time: 16 s


In [ ]:
%%time
# episcan_r9_test_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(episcan_r9_test_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    episcan_r9_test_set[model_name]=output_scores.detach().cpu().tolist()

40000it [00:01, 23215.90it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (10.9 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 14/14 [00:04<00:00,  2.92it/s]


CPU times: user 46.1 s, sys: 3.39 s, total: 49.5 s
Wall time: 7.34 s


In [ ]:
%%time
# episcan_rt_test_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(episcan_rt_test_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    episcan_rt_test_set[model_name]=output_scores.detach().cpu().tolist()

40000it [00:01, 23640.51it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (10.9 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 14/14 [00:04<00:00,  2.98it/s]


CPU times: user 46.1 s, sys: 3.55 s, total: 49.7 s
Wall time: 6.94 s


In [ ]:
%%time
# sd3_formatted
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(sd3_formatted, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    sd3_formatted[model_name]=output_scores.detach().cpu().tolist()

326579it [00:07, 41749.25it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (88.8 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 107/107 [00:17<00:00,  6.04it/s]


CPU times: user 4min 59s, sys: 10.8 s, total: 5min 10s
Wall time: 27 s


In [ ]:
%%time
# ba_full_cls_test_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ba_full_cls_test_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ba_full_cls_test_set[model_name]=output_scores.detach().cpu().tolist()

111500it [00:04, 23871.83it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (30.3 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 37/37 [00:07<00:00,  5.09it/s]


CPU times: user 1min 51s, sys: 4.54 s, total: 1min 56s
Wall time: 12.7 s


In [ ]:
%%time
# ba_full_con_test_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ba_full_con_test_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ba_full_con_test_set[model_name]=output_scores.detach().cpu().tolist()

79578it [00:01, 41686.85it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (21.6 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 26/26 [00:05<00:00,  4.42it/s]


CPU times: user 1min 20s, sys: 3.91 s, total: 1min 24s
Wall time: 8.65 s


In [ ]:
%%time
# nmp_ba_data_hla
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(nmp_ba_data_hla, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    nmp_ba_data_hla[model_name]=output_scores.detach().cpu().tolist()

170107it [00:05, 30325.31it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (46.3 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 56/56 [00:30<00:00,  1.86it/s]


CPU times: user 2min 51s, sys: 7.9 s, total: 2min 59s
Wall time: 36.7 s


In [ ]:
%%time
# escape_seq_full_cls_test_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(escape_seq_full_cls_test_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    escape_seq_full_cls_test_set[model_name]=output_scores.detach().cpu().tolist()

0it [00:00, ?it/s]

55900it [00:02, 24667.61it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (15.2 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 19/19 [00:07<00:00,  2.63it/s]


CPU times: user 58.7 s, sys: 5.11 s, total: 1min 3s
Wall time: 10.8 s


In [ ]:
%%time
# escape_seq_full_con_test_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(escape_seq_full_con_test_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    escape_seq_full_con_test_set[model_name]=output_scores.detach().cpu().tolist()

14456it [00:00, 23512.66it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (3.9 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 5/5 [00:06<00:00,  1.29s/it]


CPU times: user 17.6 s, sys: 5.91 s, total: 23.6 s
Wall time: 8.11 s


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
%%time
"""
Save test sets
"""
save_dfs=False
# save_dfs=True

if save_dfs:
    # yd test set
    holec_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "Holec_et_al_2025"
    holec_test_set.to_csv(f"{holec_datasets_dir}/holec_test_set.csv")

    # episcan test sets
    episcan_train_test_dir = DATA_DIR / "Training_Datasets" / "Processed" / "EpiScan"
    episcan_r9_test_set.to_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv")
    episcan_rt_test_set.to_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv")
    sd3_formatted.to_csv(f"{episcan_train_test_dir}/sd3_formatted.csv")
    
    # ba test sets
    ba_full_cls_test_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_cls_test_set.csv")
    ba_full_con_test_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_con_test_set.csv")
    nmp_ba_data_hla.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/nmp_ba_data_hla.csv")

    # escape-seq test sets
    escape_seq_datasets_dir = DATA_DIR / "Training_Datasets/Processed/ESCAPE_seq"
    escape_seq_full_cls_test_set.to_csv(f"{escape_seq_datasets_dir}/escape_seq_full_cls_test_set.csv")
    escape_seq_full_con_test_set.to_csv(f"{escape_seq_datasets_dir}/escape_seq_full_con_test_set.csv")


CPU times: user 3.13 s, sys: 140 ms, total: 3.27 s
Wall time: 3.26 s


### MS log odds

In [ ]:
%%time
# generate chp background
chp_filtered_negative_peptides=generate_kmers_from_proteome(canonical_human_proteins, kmer_lengths=[8,9,10,11,12,13,14])
random_negative_peptides=generate_random_peptides(n_peptides=10, lengths=[8,9,10,11,12,13,14]) 

chp_filtered_negative_peptides_no_dup={}
for l,p in tqdm(chp_filtered_negative_peptides.items()):
    chp_filtered_negative_peptides_no_dup[l]=list(set(p))
    print("Length", l)
    print(len(chp_filtered_negative_peptides[l]))
    print(len(chp_filtered_negative_peptides_no_dup[l]))

 14%|█▍        | 1/7 [00:02<00:15,  2.50s/it]

Length 8
11250173
10300888


 29%|██▊       | 2/7 [00:05<00:12,  2.51s/it]

Length 9
11229814
10405224


 43%|████▎     | 3/7 [00:07<00:10,  2.54s/it]

...

Length 12
11168738
10540392


 86%|████████▌ | 6/7 [00:15<00:02,  2.53s/it]

Length 13
11148381
10563518


100%|██████████| 7/7 [00:17<00:00,  2.53s/it]

Length 14
11128026
10580237
CPU times: user 1min 2s, sys: 5.32 s, total: 1min 7s
Wall time: 1min 7s


In [ ]:
# create cleaned peptide column
ms_train_data["peptide"]=ms_train_data.seq.str.replace("$", "")
holec_test_set["peptide"]=holec_test_set.seq.str.replace("$", "")
episcan_r9_test_set["peptide"]=episcan_r9_test_set.seq.str.replace("$", "")
episcan_rt_test_set["peptide"]=episcan_rt_test_set.seq.str.replace("$", "")
sd3_formatted["peptide"]=sd3_formatted.seq.str.replace("$", "")
ba_full_cls_test_set["peptide"]=ba_full_cls_test_set.seq.str.replace("$", "")
ba_full_con_test_set["peptide"]=ba_full_con_test_set.seq.str.replace("$", "")
nmp_ba_data_hla["peptide"]=nmp_ba_data_hla.seq.str.replace("$", "")
escape_seq_full_cls_test_set["peptide"]=escape_seq_full_cls_test_set.seq.str.replace("$", "")
escape_seq_full_con_test_set["peptide"]=escape_seq_full_con_test_set.seq.str.replace("$", "")

In [ ]:
%%time
# compute ms log odds

# get ref_df from pt set
ref_df=ms_train_data.copy()

# yd
holec_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=holec_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# episcan_r9_test_set
episcan_r9_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=episcan_r9_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# episcan_rt_test_set
episcan_rt_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=episcan_rt_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)


# sd3_formatted
sd3_formatted=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=sd3_formatted,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)


# ba_full_cls_test_set
ba_full_cls_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=ba_full_cls_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# ba_full_con_test_set
ba_full_con_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=ba_full_con_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# nmp_ba_data_hla
nmp_ba_data_hla=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=nmp_ba_data_hla,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# escape_seq_full_cls_test_set
escape_seq_full_cls_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=escape_seq_full_cls_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)


# escape_seq_full_con_test_set
escape_seq_full_con_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=escape_seq_full_con_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)


Computing PWM log-odds (allelexlen): 100%|██████████| 315/315 [00:00<00:00, 1997.69it/s]


CPU times: user 4min, sys: 31.3 s, total: 4min 31s
Wall time: 4min 31s


In [ ]:
%%time
"""
Save test sets
"""
save_dfs=False
# save_dfs=True

if save_dfs:
    # yd test set
    holec_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "Holec_et_al_2025"
    holec_test_set.to_csv(f"{holec_datasets_dir}/holec_test_set.csv")

    # episcan test sets
    episcan_train_test_dir = DATA_DIR / "Training_Datasets" / "Processed" / "EpiScan"
    episcan_r9_test_set.to_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv")
    episcan_rt_test_set.to_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv")
    sd3_formatted.to_csv(f"{episcan_train_test_dir}/sd3_formatted.csv")
    
    # ba test sets
    ba_full_cls_test_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_cls_test_set.csv")
    ba_full_con_test_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_con_test_set.csv")
    nmp_ba_data_hla.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/nmp_ba_data_hla.csv")

    # escape-seq test sets
    escape_seq_datasets_dir = DATA_DIR / "Training_Datasets/Processed/ESCAPE_seq"
    escape_seq_full_cls_test_set.to_csv(f"{escape_seq_datasets_dir}/escape_seq_full_cls_test_set.csv")
    escape_seq_full_con_test_set.to_csv(f"{escape_seq_datasets_dir}/escape_seq_full_con_test_set.csv")


CPU times: user 3.15 s, sys: 164 ms, total: 3.32 s
Wall time: 3.31 s
